In [ ]:
%pip install pandas numpy openpyxl

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

In [ ]:
# Project folder (the cleaned file and report will be saved here)
FOLDER = Path(r"C:\Users\jashw\OneDrive\Documents\2-1\DAV\DAV Project\cleaned data")
FOLDER.mkdir(parents=True, exist_ok=True)

# Find the raw dataset in this folder or anywhere inside the DAV Project folder
RAW_NAME = "indian_roads_dataset.csv"
matches = list(FOLDER.glob(RAW_NAME)) + list(FOLDER.parent.rglob(RAW_NAME))

if not matches:
    raise FileNotFoundError(
        f"Could not find {RAW_NAME} in {FOLDER.parent}. "
        f"Copy it into {FOLDER} and run this cell again."
    )

file_path = matches[0]
output_path = FOLDER / "indian_roads_dataset_cleaned.csv"
report_path = FOLDER / "indian_roads_cleaning_report.xlsx"

print("Raw file    :", file_path)
print("Cleaned file:", output_path)
print("Report file :", report_path)

In [ ]:
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("File:", Path(file_path).name)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

In [ ]:
original_df = df.copy(deep=True)

print("Original dataset backed up in memory.")
print("Original shape:", original_df.shape)

In [ ]:
print("Dataset Information:")
print("-" * 50)

print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame(name="Data_Type"))

In [ ]:
print("First 10 rows:")
display(df.head(10))

print("\nLast 5 rows:")
display(df.tail())

In [ ]:
print("Statistical summary (numeric columns):")
display(df.describe().T)

print("\nStatistical summary (text columns):")
display(df.describe(exclude="number").T)

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print("Column names standardized:")
print(df.columns.tolist())

In [ ]:
missing_before = df.isna().sum()

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Missing_Values": missing_before.values,
    "Missing_Percentage": (
        missing_before.values / len(df) * 100
    ).round(2)
})

print("Missing-value report:")
display(missing_report[missing_report["Missing_Values"] > 0])

print("Total missing cells:", int(missing_before.sum()))

In [ ]:
raw_festival = pd.read_csv(file_path, keep_default_na=False)["festival"]

print("Festival values in the raw file:")
display(
    raw_festival
    .value_counts()
    .rename_axis("festival")
    .reset_index(name="Count")
)

print("\nThe text 'None' means no festival, but pandas reads it as a missing value.")

In [ ]:
rows_before = len(df)
duplicate_rows = int(df.duplicated().sum())

df = df.drop_duplicates().copy()

rows_after = len(df)
duplicates_removed = rows_before - rows_after

print("Rows before:", rows_before)
print("Duplicate rows found:", duplicate_rows)
print("Duplicate rows removed:", duplicates_removed)
print("Rows after:", rows_after)

In [ ]:
id_columns = ["accident_id"]

for column in id_columns:
    if column in df.columns:
        duplicate_count = int(
            df[column].dropna().duplicated().sum()
        )

        print(f"{column}: {duplicate_count} repeated non-empty IDs")

        if duplicate_count > 0:
            display(
                df[df[column].duplicated(keep=False)]
                .sort_values(column)
            )

In [ ]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    df[column] = df[column].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

print("Removed leading and trailing spaces from text fields.")
print("Text columns checked:", list(text_columns))

In [ ]:
category_columns = [
    "road_type",
    "weather",
    "visibility",
    "traffic_density",
    "cause",
    "accident_severity",
    "day_of_week"
]

for column in category_columns:
    print(f"\n{column}:")
    display(
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="Count")
    )

In [ ]:
print("City and state combinations:")
display(
    df.groupby(["state", "city"])
    .size()
    .reset_index(name="Accidents")
)

cities_with_many_states = (
    df.groupby("city")["state"].nunique()
)

print(
    "Cities linked to more than one state:",
    int((cities_with_many_states > 1).sum())
)

In [ ]:
df["festival"] = df["festival"].fillna("No Festival")

print("Missing festival values replaced with 'No Festival'.")

display(
    df["festival"]
    .value_counts(dropna=False)
    .rename_axis("festival")
    .reset_index(name="Count")
)

In [ ]:
df["date"] = pd.to_datetime(
    df["date"], format="%Y-%m-%d", errors="coerce"
)

numeric_columns = [
    "latitude",
    "longitude",
    "hour",
    "is_weekend",
    "lanes",
    "traffic_signal",
    "temperature",
    "vehicles_involved",
    "casualties",
    "is_peak_hour",
    "risk_score"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

integer_columns = [
    "accident_id",
    "hour",
    "is_weekend",
    "lanes",
    "traffic_signal",
    "temperature",
    "vehicles_involved",
    "casualties",
    "is_peak_hour"
]

for column in integer_columns:
    non_missing = df[column].dropna()

    if (non_missing % 1 == 0).all():
        df[column] = df[column].astype("Int64")

print("Data type conversion completed.")
display(df.dtypes.to_frame(name="Data_Type"))

In [ ]:
missing_after_conversion = df.isna().sum()

print("Missing values after data type conversion:")
display(
    missing_after_conversion[
        missing_after_conversion > 0
    ].rename("Missing_Count").to_frame()
)

print(
    "Total missing cells after conversion:",
    int(missing_after_conversion.sum())
)

In [ ]:
time_check = pd.to_datetime(
    df["time"], format="%H:%M", errors="coerce"
)

print("Invalid time values:", int(time_check.isna().sum()))
print("Invalid date values:", int(df["date"].isna().sum()))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

In [ ]:
range_rules = {
    "latitude": (6, 38),
    "longitude": (68, 98),
    "hour": (0, 23),
    "is_weekend": (0, 1),
    "lanes": (1, 10),
    "traffic_signal": (0, 1),
    "temperature": (-10, 55),
    "vehicles_involved": (1, np.inf),
    "casualties": (0, np.inf),
    "is_peak_hour": (0, 1),
    "risk_score": (0, 1)
}

range_results = []

for column, (minimum, maximum) in range_rules.items():
    invalid_mask = (
        df[column].notna()
        & (
            (df[column] < minimum)
            | (df[column] > maximum)
        )
    )

    range_results.append({
        "Column": column,
        "Minimum_Allowed": minimum,
        "Maximum_Allowed": maximum,
        "Invalid_Count": int(invalid_mask.sum())
    })

range_report = pd.DataFrame(range_results)

print("Range validation report:")
display(range_report)

In [ ]:
allowed_values = {
    "road_type": ["highway", "rural", "urban"],
    "weather": ["clear", "fog", "rain"],
    "visibility": ["high", "low", "medium"],
    "traffic_density": ["high", "low", "medium"],
    "accident_severity": ["fatal", "major", "minor"],
    "cause": [
        "distraction", "drunk driving", "overspeeding",
        "poor road", "weather"
    ],
    "festival": ["Diwali", "Eid", "Holi", "New Year", "No Festival"]
}

category_results = []

for column, allowed in allowed_values.items():
    invalid_count = int((~df[column].isin(allowed)).sum())

    category_results.append({
        "Column": column,
        "Allowed_Values": ", ".join(allowed),
        "Invalid_Count": invalid_count
    })

category_report = pd.DataFrame(category_results)

print("Category validation report:")
display(category_report)

In [ ]:
hour_from_time = time_check.dt.hour
hour_mismatch = int((hour_from_time != df["hour"]).sum())

weekday_names = df["date"].dt.day_name()
day_mismatch = int((weekday_names != df["day_of_week"]).sum())

weekend_from_date = (df["date"].dt.dayofweek >= 5).astype(int)
weekend_mismatch = int((weekend_from_date != df["is_weekend"]).sum())

print("Hour does not match time:", hour_mismatch)
print("Day of week does not match date:", day_mismatch)
print("Weekend flag does not match date:", weekend_mismatch)

In [ ]:
outlier_columns = [
    "latitude",
    "longitude",
    "lanes",
    "temperature",
    "vehicles_involved",
    "casualties",
    "risk_score"
]

outlier_results = []

for column in outlier_columns:
    values = df[column].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_mask = (
        (df[column] < lower_bound)
        | (df[column] > upper_bound)
    )

    outlier_results.append({
        "Column": column,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "Lower_Bound": round(lower_bound, 2),
        "Upper_Bound": round(upper_bound, 2),
        "Outlier_Count": int(outlier_mask.sum())
    })

outlier_report = pd.DataFrame(outlier_results)

print("IQR outlier detection report:")
display(outlier_report)
print("Outliers are reported only; no records are removed.")

In [ ]:
# Standardize the final date and time formats
df["date"] = df["date"].dt.strftime("%d-%m-%Y")
df["time"] = pd.to_datetime(
    df["time"], format="%H:%M"
).dt.strftime("%H:%M")

print("Date format: DD-MM-YYYY")
print("Time format: HH:MM")

display(df[["date", "time", "hour"]].head())

In [ ]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Number of rows",
        "Number of columns",
        "Missing cells",
        "Duplicate full rows",
        "Duplicate accident_id values"
    ],
    "Before_Cleaning": [
        len(original_df),
        len(original_df.columns),
        int(original_df.isna().sum().sum()),
        int(original_df.duplicated().sum()),
        int(original_df["accident_id"].dropna().duplicated().sum())
    ],
    "After_Cleaning": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        int(df["accident_id"].dropna().duplicated().sum())
    ]
})

print("Before-and-after quality summary:")
display(quality_summary)

In [ ]:
final_missing_report = pd.DataFrame({
    "Column": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().mean().values * 100
    ).round(2)
})

print("Final missing-value report:")
display(final_missing_report)

In [ ]:
data_dictionary = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": [str(df[col].dtype) for col in df.columns],
    "Description": [
        {
            "accident_id": "Unique identifier for an accident",
            "city": "City where the accident happened",
            "state": "State where the accident happened",
            "latitude": "Latitude of the accident location",
            "longitude": "Longitude of the accident location",
            "date": "Date of the accident (DD-MM-YYYY)",
            "time": "Time of the accident (HH:MM)",
            "hour": "Hour of the day (0 to 23)",
            "day_of_week": "Day of the week",
            "is_weekend": "1 if the accident happened on a weekend",
            "road_type": "Type of road (highway, rural, urban)",
            "lanes": "Number of lanes on the road",
            "traffic_signal": "1 if a traffic signal is present",
            "weather": "Weather condition (clear, fog, rain)",
            "visibility": "Visibility level (high, medium, low)",
            "temperature": "Temperature in degrees Celsius",
            "traffic_density": "Traffic density (low, medium, high)",
            "cause": "Main cause of the accident",
            "accident_severity": "Severity (minor, major, fatal)",
            "vehicles_involved": "Number of vehicles involved",
            "casualties": "Number of casualties",
            "is_peak_hour": "1 if the accident happened in peak hours",
            "festival": "Festival on the day, or No Festival",
            "risk_score": "Accident risk score (0 to 1)"
        }.get(col, "Column in the dataset")
        for col in df.columns
    ]
})

print("Data dictionary created:")
display(data_dictionary)

In [ ]:
cleaned_data = df.copy()

print("Final dataset prepared.")
print("Rows:", cleaned_data.shape[0])
print("Columns:", cleaned_data.shape[1])

display(cleaned_data.head(10))

In [ ]:
cleaned_data.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Saved to:", output_path)

In [ ]:
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    quality_summary.to_excel(writer, sheet_name="Quality_Summary", index=False)
    final_missing_report.to_excel(writer, sheet_name="Missing_Values", index=False)
    range_report.to_excel(writer, sheet_name="Range_Checks", index=False)
    category_report.to_excel(writer, sheet_name="Category_Checks", index=False)
    outlier_report.to_excel(writer, sheet_name="Outlier_Report", index=False)
    data_dictionary.to_excel(writer, sheet_name="Data_Dictionary", index=False)

print("Cleaning report saved successfully!")
print("Saved to:", report_path)

In [ ]:
if Path(output_path).exists():
    verification_df = pd.read_csv(output_path)

    print("Cleaned file verified successfully!")
    print("Saved file:", Path(output_path).name)
    print("Final dataset dimensions:", verification_df.shape)
    print("Missing cells:", int(verification_df.isna().sum().sum()))

    display(verification_df.head())
else:
    print("No cleaned file found. Run the save cell first.")

if Path(report_path).exists():
    check = pd.ExcelFile(report_path)

    print("\nReport sheets available:")

    for sheet in check.sheet_names:
        print("-", sheet)
else:
    print("No report file found. Run the report cell first.")